# RDSGD Theoretical Bounds Tightness Check
Is the empirical generalization gap bounded above as described in theorem 13?
Is the optimality gap as predicted in Lemma 11?

In [1]:
%load_ext autoreload
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path
from fastnanoid import generate
from datetime import datetime

In [2]:
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
lib_path = project_root / "lib"
sys.path.insert(0, str(lib_path))        
sys.path.insert(0, str(project_root))   

In [3]:
%autoreload 2
from lib.graph_factory import GraphFactory
from lib.proj_const_estimator import ProjConstEstimator
from lib.utils import fetch_dataset, get_alphas, rng, seed
from lib.classifier import ByzClassifier
from lib.system import SystemSimulator
from lib.config import BASE_CONF, LOGFILE, NUM_NODES

In [4]:
RUN_DIR = Path().resolve()
config = BASE_CONF

In [5]:
global_dataset = fetch_dataset('MNIST')
gf = GraphFactory(config['train']['num_nodes'], config['train']['b'], seed('graph'))
proj_const_estimator = ProjConstEstimator(config, global_dataset, rng('proj-const-estimator')) 
classifier = ByzClassifier(config, global_dataset, rng('byz-classifier'))
sys_sim = SystemSimulator(config, global_dataset, rng('ml-system-sim'), LOGFILE)

In [6]:
proj_const_estimator.configure(gf,config)
proj_const = proj_const_estimator.estimate()

In [7]:
%%time
classifier.configure(gf,config, proj_const)
X_C_train, y_C_train, X_C_test, y_C_test = classifier.run_simulations()
classifier_metrics = classifier.train_and_eval(X_C_train, y_C_train, X_C_test, y_C_test)
C_opt_pt = classifier.clac_opt_operating_pt(0.7)
C_best_est, C_pre_pipe = classifier.get_params()

CPU times: user 6min 8s, sys: 7.06 s, total: 6min 15s
Wall time: 31.5 s


In [11]:
%%time
sim_params = {
'algorithms':['RDSGD'],
'atk_type':config['sys']['atk_type'],
'threat_model':'T3'
}

run_results = dict()
RUN_ID = generate()
run_results['timestamp'] = pd.to_datetime(datetime.now())
run_results['proj_const'] = proj_const
run_results.update(C_opt_pt)

sys_sim.configure(gf, config, proj_const, C_pre_pipe, C_best_est, C_opt_pt)
df = sys_sim.simulate(sim_params, run_results)
payload = sys_sim.log_results(run_results, RUN_DIR, RUN_ID)

==========RDSGD==========
[k=35] train_loss=1.6342
[k=70] train_loss=1.6049
[k=105] train_loss=1.5973
CPU times: user 13min 5s, sys: 4.35 s, total: 13min 10s
Wall time: 37.5 s


In [26]:
payload['opt_gap_emp'] <= payload['opt_gap_ub'] # lemma 11

np.True_

In [27]:
df.loc['RDSGD']['Ck'].iloc[-1] <= payload['byz_cons_floor'] # Lyapunov upper-bound of V^inf (corollary 10)

np.True_